In [1]:
from transformers import BertTokenizer, BertModel
import torch
from datasets import Dataset
import pandas as pd
import lightgbm as lgb
import sklearn
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_curve
from sklearn.multioutput import MultiOutputClassifier


In [7]:
#Getting text input
df = pd.read_csv("/content/drive/MyDrive/data_preprocessed.csv")
data = df[['Cargo', 'Empresa', 'Local', 'Descricao']].values.tolist()
print(data[0])

['Recepcionista', 'SELECT RH', 'Belo Horizonte - MG', 'HORÁRIO : \n  Seg Sex    09:00 18:00 / Sáb    09:00 13:00 \n  Atividades principal : \n  COBRANÇA de TAXA AJUSTE DE LANÇAMENTOS ATUALIZAÇÃO CADASTRAL ARQUIVAÇÃO de CARTAS de APRESENTAÇÃO RESPONSÁVEL PARTE CONDOMÍNIO \n   benefício : \n    Valerefeição \n    Valetransporte']


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
#Loading and tokenizing Input
tokenizer = BertTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')
bertimbau = BertModel.from_pretrained('neuralmind/bert-base-portuguese-cased')

def extractFeatures():
  result = None
  token = tokenizer(df['Descricao'].tolist(), return_tensors='pt', truncation=True, padding=True, max_length=512)
  with torch.no_grad():
    output = bertimbau(**token)
    encoded = output[0][:, 1:-1]
    return encoded

  for i in range(0,501):
     #token = tokenizer(df.at[i, 'Cargo'],df.at[i, 'Empresa'],df.at[i, 'Descricao'], return_tensors='pt', truncation=True, padding=True, max_length=512)
     with torch.no_grad():
        output = bertimbau(**token)
        encoded = output[0][:, 1:-1]
     if result is None:
        result = encoded
     else:
        print(result.shape)
        print(encoded.shape)
        result = torch.cat((result, encoded), dim = 0)

  return result

input = extractFeatures()
input.shape


tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

torch.Size([501, 510, 768])

In [13]:
y_df = df.drop(['Cargo', 'Empresa', 'Local', 'Descricao', 'Salario Coletado', 'Salario', 'Media', 'Qtd Vagas', 'Unnamed: 0'], axis=1)

X = torch.reshape(X, (501, 510*768))
y = torch.tensor(y_df.values.tolist())

X = X[:,:1000]
X.shape, y.shape

(torch.Size([501, 1000]), torch.Size([501, 11]))

Kfold


In [ ]:
scores = []
recalls = []
f1scores = []
rocs = []

def kfold_cross_validation(splits, shuffle, rng_state=42, batch_size = 1):
  kf = KFold(n_splits=splits, shuffle= shuffle, random_state=rng_state)
  for train_index, val_index in kf.split(X):
    X_train, X_val = X[train_index], X[val_index]

    y_train, y_val = y[train_index], y[val_index]

    lgb_model = lgb.LGBMClassifier(learning_rate=0.05, n_estimators=50)
    model = MultiOutputClassifier(lgb_model)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    score = accuracy_score(y_val, y_pred)
    scores.append(score)

    recall = recall_score(y_val, y_pred, average='micro')
    recalls.append(recall)

    f1 = f1_score(y_val,y_pred, average='micro')
    f1scores.append(f1)
    #curRoc.append(roc_curve(y_val,y_pred))

kfold_cross_validation(10,True,42,25)
np.mean(scores), np.mean(recalls), np.mean(f1scores), np.mean(rocs)
scores,recalls,f1scores,rocs



[LightGBM] [Info] Number of positive: 24, number of negative: 426
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010460 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 150318
[LightGBM] [Info] Number of data points in the train set: 450, number of used features: 1000
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.053333 -> initscore=-2.876386
[LightGBM] [Info] Start training from score -2.876386
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

[LightGBM] [Info] Number of positive: 28, number of negative: 423
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011368 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 150655
[LightGBM] [Info] Number of data points in the train set: 451, number of used features: 1000
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.062084 -> initscore=-2.715168
[LightGBM] [Info] Start training from score -2.715168
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 